## IMPORTATION DES BIBLIOTHEQUES ET PARAMETRES GLOBAUX

In [1]:
### Importation des bibliothèques standards et paramètres globaux ###
import os
import json
import importlib
import time
from datetime import datetime

# --- Bibliothèques scientifiques ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import LinearSegmentedColormap

# --- Configuration de l'affichage matplotlib ---
%matplotlib qt 
# Mode interactif pour les figures

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 20
})

# --- Options d'affichage Pandas ---
pd.set_option('display.max_rows', 100)
pd.options.display.float_format = '{:,.2f}'.format

#pd.options.display.float_format = '{:,.2f}'.format  # Affichage avec séparateur de milliers

# ==============================
# PARAMÈTRES GLOBAUX
# ==============================

# Obtenir la date du jour au format jjmmaaa
date_du_jour = datetime.now().strftime("%d%m%y")


### Importation des fonctions définies dans les fichiers biodiv ###

In [2]:

import affichage_carte_biodiv
importlib.reload(affichage_carte_biodiv)
from affichage_carte_biodiv import (
    configurer_carte,
    ajouter_couche_SIG,
    ajouter_couche_continue,
    ajouter_couche_discrete,
    ajouter_couche_statut,
    ajouter_couche_point,
    afficher_carte_defaut,
    afficher_fond_carte,
    afficher_carte_interactive
)
import normalisation_biodiv
importlib.reload(normalisation_biodiv)
from normalisation_biodiv import (
    normaliser_par_maille,
    normaliser_par_maille_et_clade,
    normaliser_par_aire,
    normaliser_par_aire_et_clade,
    normaliser_par_espece,
    normaliser_par_clade,
    normaliser_unique,
    normaliser_log,
    normaliser_par_periode
)
import exploration_biodiv
importlib.reload(exploration_biodiv)
from exploration_biodiv import (
    filtrer_top,
    afficher_top_especes,
    chercher_espece,
    explorer_clade,
    chercher_especes_protegees
)
import correlation_prediction
importlib.reload(correlation_prediction)
from correlation_prediction import (
    calculer_matrice_correlation,
    calculer_correlation_sujet,
    recalculer_nombreObs_par_correlation,
    calculer_prediction,
    calculer_seuil,
    recherche_espece_absente,
    prepare_data,
    fit_and_plot,
    plot_residuals,
    appliquer_transformation
)
import biodiversite_endemisme_biodiv
importlib.reload(biodiversite_endemisme_biodiv)
from biodiversite_endemisme_biodiv import (
    calculer_shannon,
    calculer_simpson,
    calculer_WE,
    calculer_indices
)
import clustering_geo_biodiv
importlib.reload(clustering_geo_biodiv)
from clustering_geo_biodiv import (
    analyser_composantes_principales,
    former_cluster_biogeo,
    determiner_k,
    etudier_composition_cluster,
    kmeans_with_spatial_constraint,
    compute_spatial_centroids,
    assign_missing_clusters,
    former_cluster_biogeo_avec_critere_spatial,
    calculer_inertie
)

import clustering_espece_biodiv
importlib.reload(clustering_espece_biodiv)
from clustering_espece_biodiv import (
    generer_dendogram,
    former_cluster_espece,
    chercher_numcluster_espece,
    lister_especes_dans_cluster,
    grouper_par_cluster,
    etudier_un_cluster_local,
    rechercher_especes_localement_absentes
)

import fonctions_annexes_biodiv
importlib.reload(fonctions_annexes_biodiv)
from fonctions_annexes_biodiv import (
    round_to_sig,
    generer_dictionnaire_taxonomie,
    afficher_dataframe,
    completer_df,
    lister_mailles_dans_site,
    ajouter_nom_site_df,
    filtrer_grille,
    afficher_carte_monde,
    filtrer_geo,
    filtrer_categorie
)

import evolution_temporelle
importlib.reload(evolution_temporelle)
from evolution_temporelle import (
    suivre_disparition_geo,
    determiner_statut
)

## IMPORTATION DES DONNEES

In [3]:
# Définition des régions par continent
regions = {
    "Amerique_centrale": ['Panama', 'Costa Rica', 'Nicaragua', 'El Salvador', 'Honduras', 
                          'Guatemala', 'Belize', 'Colombia', 'Mexico', 'Cuba'],
    "Amerique_Sud": ['Colombia', 'Brazil', 'Argentina', 'Peru', 'Ecuador', 'Chile', 
                     'Bolivia', 'Paraguay', 'Uruguay', 'Venezuela'],
    "Asie_du_sud": ['India', 'Sri Lanka', 'Bangladesh', 'Nepal', 'Bhutan'],
    "Asie_de_est": ['Japan', 'Republic of Korea'],
    "Asie_du_sud_est": ["Lao People's Democratic Republic", "Myanmar", "Thailand", 
                        "Malaysia", "Vietnam", "Cambodia", "Philippines"],
    "Europe": ["Spain", "Italy", 'Slovenia', 'Switzerland', 'Croatia', 'Portugal', "Austria", 
               "France", "Germany", "Greece", "Slovakia", "Czechia", "Poland", "Lithuania", 
               "Latvia", "Estonia", "Finland", "Hungary", 'Romania', 'Ukraine', 'Bulgaria',
               'North Macedonia', 'Bosnia and Herzegovina', 'Serbia', 'Kosovo', 'Albania', 
               'Moldova', 'Turkey', 'Belarus', 'Montenegro', 'Cyprus'],
    "Med": ["Spain", "France", 'Italy', 'Slovenia', 'Croatia', 'Greece', "Turkey", 
            "Tunisia", "Algeria", "Morocco"],
    "Maghreb": ["Tunisia", "Algeria", "Morocco"],
    "East_africa": ['Ethiopia', 'Kenya', 'Uganda', 'Tanzania', 'Rwanda', 'Burundi', 'South Sudan']
}

### Importation des données

In [4]:
# Choix du mode d'importation : "local" ou "pays"
mode_importation = "local"  # Changer en "local" pour les données locales
countries = ["France"] # format pays : regions["Asie_du_sud"] , format local :['France']
zone_name_short="Sud de la France"
ecosysteme = "terrestre" # terrestre, maritime, combined

cle_ID = "speciesKey"

grid_size_km = 5  # Par défaut, mais peut être changé pour local
cle_geo = f"codeMaille{grid_size_km}Km"

bornes_temporelles = [1800, 1990, 2010, 2024]
# Créer une chaîne de caractères avec les bornes séparées par "_"
chaine_bornes = "_".join(map(str, bornes_temporelles))

if mode_importation=="local":
    # Définition des zones locales
    zones_locales = ["66", "11", "81", "34", "12", "48", "30", "07", "26", "84", "13", "05", "04", "83", "06"]
    zone_name = "_".join(countries + zones_locales)

if mode_importation=="pays":
    zone_name = "_".join(countries)

print(f"Mode choisi : {mode_importation} // nom de la zone : {zone_name_short} // ecosysteme : {ecosysteme} // nom du fichier : {ecosysteme}")

# Définition des chemins
base_path = r'D:\MANTIS'
data_path = os.path.join(base_path, 'Data')
save_path = os.path.join(base_path, 'Resultats', f'results_{date_du_jour}')

# Vérification et création du dossier de sauvegarde
os.makedirs(save_path, exist_ok=True)
print(f"Répertoire {'existant' if os.path.exists(save_path) else 'créé'} : {save_path}")

# INITIALISATION DES DATAFRAMES
df_raw_import = pd.DataFrame()
carte_maille = pd.DataFrame()
border_local_geo = gpd.GeoDataFrame()
eez_local_geo = gpd.GeoDataFrame()

# CHARGEMENT DES DONNÉES GÉOGRAPHIQUES GLOBALES
border_geo = gpd.read_file(os.path.join(data_path, "SIG_Global", "world-administrative-boundaries.geojson"))
eez_geo = gpd.read_file(os.path.join(data_path, "SIG_Global", "eez_v11.gpkg"))
chemin_dico_cartes = os.path.join(data_path, "SIG_Global", "dictionnaire_cartes.json")

with open(chemin_dico_cartes, "r", encoding="utf-8") as fichier:
    dictionnaire_cartes = json.load(fichier)

print("✅ Importation des données géographiques globales")


# IMPORTATION DES DONNÉES
cle_geo = f"codeMaille{grid_size_km}Km"

for country in countries:
    print(f"📥 Importation des données pour {zone_name}...")

    country_code = border_geo.loc[border_geo["name"] == country, "color_code"].iloc[0]

    # Chargement des frontières et zones économiques exclusives
    border_local_geo = pd.concat([border_local_geo, border_geo[border_geo["color_code"] == country_code]], ignore_index=True)
    eez_local_geo = pd.concat([eez_local_geo, eez_geo[eez_geo["ISO_SOV1"] == country_code]], ignore_index=True)

    # Définition du chemin des données
    country_safe = country.replace(" ", "_")
    path_data_country = os.path.join(data_path,"GBIF", f"GBIF_{country_safe}")

    # Chargement des données géographiques des mailles
    if mode_importation == "pays":
        fichier_grille_local = os.path.join(path_data_country, "SIG", f"grid_{country_safe}_{ecosysteme}_{cle_geo}.geojson")   
    else:
        fichier_grille_local = os.path.join(path_data_country, "SIG", f"France_66_11_81_34_12_48_30_07_26_84_13_05_04_83_06_grid_5km.geojson")

    if os.path.exists(fichier_grille_local):
        carte_maille = pd.concat([carte_maille, gpd.read_file(fichier_grille_local)], ignore_index=True)
    else:
        print(f"⚠️ Fichier manquant : {fichier_grille_local}")

    # Chargement des données GBIF
    if mode_importation == "pays":
        fichier_gbif = os.path.join(path_data_country,'processed', f"data_GBIF_{country_safe}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}.csv")
    else:
        fichier_gbif = os.path.join(path_data_country,'processed', f"data_GBIF_{zone_name}_{cle_geo}.csv")

    if os.path.exists(fichier_gbif):
        df_temp = pd.read_csv(fichier_gbif, dtype={"nombreObs": int, cle_ID: int})
        df_raw_import = pd.concat([df_raw_import, df_temp], ignore_index=True)
    else:
        print(f"⚠️ Fichier manquant : {fichier_gbif}")

print("✅ Importation terminée pour tous les pays/zones.")

# ==============================
# TRAITEMENT DES DONNÉES
# ==============================

# Filtrage des mailles valides
df_raw_import = df_raw_import[df_raw_import[cle_geo].isin(carte_maille[cle_geo])]
print("✅ Filtrage des mailles valides")

# Génération du dictionnaire taxonomique
dico_taxo = generer_dictionnaire_taxonomie(df_raw_import, cle_ID)
liste_col_taxo=dico_taxo.columns.tolist()

# Agrégation par période et clé
df_raw_import = df_raw_import.groupby([cle_geo, cle_ID, "periode"], observed=True)["nombreObs"].sum().reset_index()

# Fusion avec la taxonomie
df_raw_import = df_raw_import.merge(dico_taxo, on=cle_ID, how="left")
print("✅ Génération du dictionnaire taxonomique")


print("✅ Importation et traitement terminés")


Mode choisi : local // nom de la zone : Sud de la France // ecosysteme : terrestre // nom du fichier : terrestre
Répertoire existant : D:\MANTIS\Resultats\results_100725
✅ Importation des données géographiques globales
📥 Importation des données pour France_66_11_81_34_12_48_30_07_26_84_13_05_04_83_06...
✅ Importation terminée pour tous les pays/zones.
✅ Filtrage des mailles valides
✅ Génération du dictionnaire taxonomique
✅ Importation et traitement terminés


In [5]:
# Filtrage des données et statistiques

# Filtrage géographique si nécessaire

filtrage_geo = False
carte_maille_saved=carte_maille.copy()
if filtrage_geo:
    zone_name_short="Sicily"
    afficher_carte_monde(border_geo, 15)
    lon_min, lon_max = 11, 15.65
    lat_min, lat_max = 36.6, 39
    filtrage_geo = input("Souhaitez-vous filtrer la carte géographiquement ? (o/n) ").lower() == 'o'
    carte_maille= filtrer_grille(carte_maille_saved, lat_min, lat_max, lon_min, lon_max)
    df_raw_import = df_raw_import[df_raw_import[cle_geo].isin(carte_maille[cle_geo])].copy()
# Afficher le nombre de mailles 
print(f"Nombre de mailles dans la zone selectionnées : {carte_maille[cle_geo].nunique()}")
print(f"\nNombre de mailles avec données : {df_raw_import[cle_geo].nunique()}")
print("\n")


# Afficher les règnes existants
regnes_disponibles = df_raw_import['kingdom'].unique()
print("Règnes disponibles :", regnes_disponibles)

# Définir les règnes à inclure (True pour inclure, False pour exclure)
selection_regnes = {
    'Animalia': True,
    'Plantae': True,
    'Fungi': False,
    'Protozoa': False,
    'Chromista': False,
    'Bacteria': False,
    'Viruses': False,
    'Archaea': False# Modifier ici selon les besoins
    }

# Filtrer les données en fonction des règnes sélectionnés
regnes_a_garder = [regne for regne, garder in selection_regnes.items() if garder]
df_raw_filtred = df_raw_import[df_raw_import['kingdom'].isin(regnes_a_garder)].copy()
# Affichage des règnes filtrés
print("Règnes sélectionnés :", regnes_a_garder)

# Compter le nombre d'especes
df_raw_filtred = normaliser_unique(df_raw_filtred)  

# Calcul des statistiques
stats = df_raw_filtred.groupby(cle_geo)[['nombreObs', 'nombreObs_unique']].sum()

# Renommer la colonne 'nombreObs_unique' en 'nombreEspeces'
stats.rename(columns={'nombreObs': 'nombreObs par maille'}, inplace=True)
stats.rename(columns={'nombreObs_unique': 'nombreEspeces par maille'}, inplace=True)

# Création du tableau de synthèse
stats_summary = pd.DataFrame({
    "Moyenne": stats.mean(),
    "Médiane": stats.median(),
    "1er décile": stats.quantile(0.1),
    "9e décile": stats.quantile(0.9)
}).astype(int)
print("\n")
# Afprint(stats_summary)fichage sous forme de tableau
print(stats_summary)

# Copie des données avec périodes
df_biodiv_periode = df_raw_filtred.copy()

# Création du DataFrame sans période (somme globale)
df_biodiv_sansperiode = df_biodiv_periode.groupby([cle_geo, cle_ID], as_index=False)["nombreObs"].sum()
df_biodiv_sansperiode = df_biodiv_sansperiode.merge(dico_taxo, on=cle_ID, how="left")


Nombre de mailles dans la zone selectionnées : 3748

Nombre de mailles avec données : 3748


Règnes disponibles : ['Animalia' 'Plantae' 'Fungi' 'Chromista' 'Protozoa' 'Bacteria' 'Archaea'
 'incertae sedis' 'Viruses']
Règnes sélectionnés : ['Animalia', 'Plantae']


                          Moyenne  Médiane  1er décile  9e décile
nombreObs par maille        11396     6382        1773      25387
nombreEspeces par maille     1455     1248         540       2651


In [6]:
# Supposons que vous avez déjà un GeoDataFrame "carte_maille"
map_interactive = afficher_carte_interactive(carte_maille,cle_geo)
#map_interactive.save("ma_carte.html")  # Pour l'enregistrer
map_interactive  # Dans un notebook Jupyter, cela l'affiche directement

# 10kmE00694N04801FRA 10kmE00707N04801FRA

In [7]:
# Chargement de fichiers SIG spécifiques à la France

# Chemins des fichiers SIG
bioregion_fichier = os.path.join(data_path,"GBIF",'GBIF_France', "SIG", "region_biogeographique.shp")
departement_fichier = os.path.join(data_path,"GBIF",'GBIF_France', "SIG", "carte_departements.geojson")
PNR_fichier = os.path.join(data_path,"GBIF",'GBIF_France', "SIG", "N_ENP_PNR_S_000.shx")
PN_fichier = os.path.join(data_path,"GBIF",'GBIF_France', "SIG", "N_ENP_PN_S_000.shx")

# Chargement des fichiers SIG
bioregion_gpd = gpd.read_file(bioregion_fichier)  # Régions biogéographiques
departement_gpd = gpd.read_file(departement_fichier)  # Départements
PNR_gpd = gpd.read_file(PNR_fichier)[['NOM_SITE', 'geometry']]  # Parcs Naturels Régionaux
PN_gpd = gpd.read_file(PN_fichier)[['NOM_SITE', 'geometry']]  # Parcs Nationaux
print("✅ Chargement des fichiers SIG terminés.")

# Suppression des mentions "aire d'adhésion" dans les noms des Parcs Nationaux
PN_gpd["NOM_SITE"] = PN_gpd["NOM_SITE"].str.replace(r"\s*\[aire d'adhésion\]\s*", "", regex=True)
PN_gpd["NOM_SITE"] = PN_gpd["NOM_SITE"].str.replace(r"\s*\[Aire d'adhésion\]\s*", "", regex=True)

# Fusionner les géométries par nom de site (dissolve)
PN_gpd_fusionne = PN_gpd.dissolve(by="NOM_SITE")

# Réinitialisation de l'index après fusion
PN_gpd_fusionne.reset_index(inplace=True)

# Fusionner les Parcs Nationaux et Régionaux dans un même GeoDataFrame
PN_et_PNR_gpd = pd.concat([PNR_gpd, PN_gpd_fusionne], axis=0)
PN_et_PNR_gpd = PN_et_PNR_gpd.sort_values(by='NOM_SITE', ascending=True)
PN_et_PNR_gpd = PN_et_PNR_gpd.reset_index(drop=True)



✅ Chargement des fichiers SIG terminés.


In [25]:
#Filtrer les données selon un espace géographique
filtrage_geo = False
filtrage_site=False
type_site=PN_et_PNR_gpd
site = ['Ballons des Vosges']
cle_nom_site='NOM_SITE'
taux_min=0.01

liste_mailles = []

if filtrage_site:  # Vérifie que la liste n'est pas vide
    for s in site:
        mailles = lister_mailles_dans_site(
            carte_maille, type_site, s,
            taux_min=taux_min, cle_geo=cle_geo, cle_nom_site=cle_nom_site, methode='exact'
        )
        liste_mailles.extend(mailles)

    # Élimine les doublons si nécessaire
    liste_mailles = list(set(liste_mailles))
        
if filtrage_geo:
    df_biodiv = df_raw_import[df_raw_import[cle_geo].isin(liste_mailles)]
    carte_maille=carte_maille[carte_maille[cle_geo].isin(liste_mailles)]


zones_locales = site

if zones_locales:
    Zone_gpd_filtered = type_site[type_site[cle_nom_site].isin(zones_locales)]



In [8]:
liste_mailles = ['10kmE00694N04801FR', '10kmE00707N04801FRA']
df_biodiv_vosges = df_biodiv[df_biodiv[cle_geo].isin(liste_mailles)]

###  Normalisation des données

In [8]:

def normaliser_all_in_one(df_biodiv,carte_maille, cle_ID, cle_geo,size_grid, col='nombreObs'):
    """
    Fonction pour filtrer et normaliser les données de biodiversité.
    
    Paramètres :
    - df_biodiv : DataFrame des observations
    - cle_ID : Clé d'identification des espèces
    - cle_geo : Clé géographique des mailles
    - col : Colonne à normaliser (par défaut 'nombreObs')

    Retourne :
    - df_biodiv : DataFrame normalisé
    """

    # Normalisation initiale des données
    df_biodiv_norm = normaliser_unique(df_biodiv)  

    # Normalisation selon différentes échelles
    df_biodiv = normaliser_par_espece(df_biodiv, cle_ID, col)
    df_biodiv_norm=normaliser_par_aire_et_clade(df_biodiv_norm, carte_maille, cle_geo=cle_geo,size_grid=size_grid, clade_col='species', observation_col=col)
    df_biodiv_norm=normaliser_par_aire_et_clade(df_biodiv_norm, carte_maille, cle_geo=cle_geo,size_grid=size_grid, clade_col='class', observation_col=col)
    df_biodiv_norm=normaliser_par_aire_et_clade(df_biodiv_norm, carte_maille, cle_geo=cle_geo,size_grid=size_grid, clade_col='kingdom', observation_col=col)
    #df_biodiv_norm=normaliser_par_maille_et_clade(df_biodiv_norm, cle_geo=cle_geo, clade_col='kingdom', observation_col=col)
    
    # Normalisation logarithmique
    df_biodiv_norm = normaliser_log(df_biodiv_norm, 'nombreObs_norm_par_maille_et_kingdom')
    df_biodiv_norm = normaliser_log(df_biodiv_norm, col)

    # Extraction des colonnes contenant 'nombreObs'**
    liste_nombres = [col for col in df_biodiv_norm.columns if 'nombreObs' in col]

    return df_biodiv_norm

# Application de la fonction
df_biodiv = normaliser_all_in_one(df_biodiv_sansperiode,carte_maille, cle_ID, cle_geo,size_grid=grid_size_km)

print("✅ Normalisations terminées")

Le nombre d observations total par espèce est fixé à 10 000


D:\MANTIS\Code\normalisation_biodiv.py:98: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  carte_maille['aire'] = carte_maille.geometry.area


Les aires sont normalisées à 5 km2.
Il y a en moyenne 0.4 observations par species et par km2.
Les aires sont normalisées à 5 km2.


D:\MANTIS\Code\normalisation_biodiv.py:98: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  carte_maille['aire'] = carte_maille.geometry.area


Il y a en moyenne 27.9 observations par class et par km2.
Les aires sont normalisées à 5 km2.


D:\MANTIS\Code\normalisation_biodiv.py:98: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  carte_maille['aire'] = carte_maille.geometry.area


Il y a en moyenne 228.0 observations par kingdom et par km2.
✅ Normalisations terminées


In [9]:
# Définition des filtres sous forme de dictionnaire
filtres = {
    "arbres": {"class": ["Pinopsida"], "order": ["Fagales"], "genus": ["Crataegus", "Prunus", "Malus", "Sorbus", "Pyrus"]},
    "arbres_reduit": {"class": ["Pinopsida"], "order": ["Fagales"]},
    "plantes": {"kingdom": ["Plantae"]},
    "mammifères": {"class": ["Mammalia"]},
    "poissons": {"class": ["Actinopterygii", "Myxini", "Leptocardii", "Holocephali", "Dipneusti", "Petromyzonti", "Elasmobranchii"],
                 "order": ["Perciformes", "Tetraodontiformes", "Clupeiformes", "Characiformes", "Syngnathiformes", "Beloniformes", "Gymnotiformes"]},
    "oiseaux": {"class": ["Aves"]},
    "reptiles": {"class": ["Chelonii", "Squamata", "Crocodylia"]},
    "arthropodes": {"class": ["Insecta", "Arachnida", "Malacostraca", "Branchiopoda", "Chilopoda", "Diplopoda"]},
    "pollinisateurs": {"order": ["Hymenoptera", "Lepidoptera", "Diptera"]},
    "graminées": {"family": ["Poaceae"]},
    "mousses": {"order": ["Sphagnales", "Hypnales", "Dicranales", "Bryales", "Buxbaumiales", "Diphysciales", "Grimmiales",
                          "Andreaeales", "Polytrichales", "Hookeriales"]},
    "amphibiens": {"class": ["Amphibia"]},
    "mollusques": {"class": ["Gastropoda", "Bivalvia", "Cephalopoda", "Monoplacophora", "Scaphopoda"]},
    "araignées": {"order": ["Araneae"]},
    "champignons": {"kingdom": ["Fungi"]},
    "orchidées": {"family": ["Orchidaceae"]},
    "papillons": {"order": ["Lepidoptera"]}
}

# Exemple avec df_biodiv et filtre 'arbres'
filtre='arbres_reduit'
df_filt = filtrer_categorie(df_biodiv, filtre,filtres)
print(f"{df_filt['nombreObs'].sum()} observations de {filtre}")
print(f"{df_filt[cle_ID].nunique()} espèces de {filtre}")

905956 observations de arbres_reduit
147 espèces de arbres_reduit


In [11]:
df_biodiv_vosges = df_biodiv[df_biodiv[cle_geo].isin(liste_mailles)]

## EXPLORATION DES DONNEES  

In [10]:
#Calculer la surface de la zone maillée en km2
carte_maille = carte_maille.to_crs(epsg=4326)
carte_maille['aire'] = carte_maille.geometry.area
    
max_aire = carte_maille['aire'].max()  # Trouver la valeur maximale de l'aire
carte_maille['aire'] = (carte_maille['aire'] / max_aire) * (grid_size_km ** 2)
print(f"La superficie du territoire maillé est de {round(carte_maille['aire'].sum())} km2.")


La superficie du territoire maillé est de 93700 km2.


C:\Users\User 1\AppData\Local\Temp\ipykernel_2136\1183355384.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  carte_maille['aire'] = carte_maille.geometry.area


### Afficher le top des espèces les plus observées

In [11]:
# Sélectionner les données
df_filt = df_biodiv.copy()
df_filt=filtrer_top(df_filt, 'nombreObs', 10, cle_ID)
df_filt = filtrer_categorie(df_filt, 'orchidées',filtres)
#"df_filt=df_filt[df_filt['family']=='Euphorbiaceae']
#df_filt = df_biodiv[df_biodiv['order'] == 'Hymenoptera']

# Afficher le nombre d'espèces uniques observées
print(f"{df_filt['nombreObs'].sum()} observations")
print(f"{df_filt[cle_ID].nunique()} espèces")

# Définir la colonne des valeurs
col_valeur = 'nombreObs'

# Afficher le top des espèces les plus observées
top_espece = afficher_top_especes(df_filt, dico_taxo, col_valeur, cle_ID)
afficher_dataframe(top_espece, [col_valeur] + liste_col_taxo, col_valeur).head(50)


nombre d'espèces retenues dans le df :21732 (54%)
891814 observations
128 espèces


,nombreObs,speciesKey,species,vernacularName_fr,vernacularName_en,genus,family,order,class,phylum,kingdom,occurrenceID
0,62620,5324959,Himantoglossum robertianum,NaN,NaN,Himantoglossum,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,89ba5fee-5185-11ea-9f2f-005056968749
1,54340,2793334,Ophrys sphegodes,NaN,Early Spider-Orchid,Ophrys,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,b9536085-120b-4364-b590-a98ca3e23f27
2,49558,2808330,Anacamptis pyramidalis,Orchis Pyramidal,Pyramidal Orchid,Anacamptis,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,cf8b0acb-e41d-42ad-9059-7e931ce18b2c
3,47718,8794742,Orchis purpurea,Orchis Pourpré,Lady Orchid,Orchis,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,c0365966-ce36-4b0d-8f7f-72910d9900d5
4,40062,5324951,Himantoglossum hircinum,Orchis Bouc,Lizard Orchid,Himantoglossum,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,89b94528-5185-11ea-9f2f-005056968749
5,25513,5312179,Limodorum abortivum,NaN,NaN,Limodorum,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,1f9338dc-6560-446e-846d-bb6cd7898dd5
6,25450,8179794,Dactylorhiza maculata,Orchis Tacheté,Heath Spotted-Orchid,Dactylorhiza,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,https://www.inaturalist.org/observations/11919...
7,24854,2795869,Cephalanthera longifolia,Céphalanthère À Feuilles En Épée,Narrow-Leaved Helleborine,Cephalanthera,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,b16d9fde-0c60-4232-852a-6ee9f91a3876
8,24565,7845022,Gymnadenia conopsea,Gymnadénie Moucheron,Fragrant Orchid,Gymnadenia,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,https://www.inaturalist.org/observations/51137816
9,24372,2816250,Neottia ovata,Grande Listère,Common Twayblade,Neottia,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,c5bb84fe-4a89-4fcc-bc9f-c3e455a41528


### Chercher les espèces à partir d'un mot 

In [15]:

df_filt = filtrer_categorie(df_biodiv, 'plantes',filtres)
#df_filt=df_filt[df_filt['class']=='Insecta']
mot="Colchicum"

col_valeur='nombreObs'

resultat_recherche=chercher_espece(df_filt,dico_taxo,mot,col_valeur,cle_ID)

print(f"Nombre d'espèces correspondant au critère : {len(resultat_recherche)} espèces")
print(f"Nombre d'observations correspondant au critère : {resultat_recherche['nombreObs'].sum()} observations")

# Afficher les espèces contenant le mot cherché
afficher_dataframe(resultat_recherche,[col_valeur]+liste_col_taxo,col_valeur).head(50)

Nombre d'espèces correspondant au critère : 12 espèces
Nombre d'observations correspondant au critère : 18716 observations


,nombreObs,speciesKey,species,vernacularName_fr,vernacularName_en,genus,family,order,class,phylum,kingdom,occurrenceID
0,10047,2739622,Colchicum autumnale,Colchique Dautomne,Meadow Saffron,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,c54536b7-dbc7-4825-b4bf-2053f9ea8581
1,3595,2739801,Colchicum longifolium,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/66f1bebf-5fe6-42...
2,1908,2739836,Colchicum alpinum,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/257289a7-70f2-41...
3,1702,2739661,Colchicum bulbocodium,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,http://pifh.fr/occtax/ce1740c2-269a-46be-81b9-...
4,1278,2739964,Colchicum filifolium,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/b0be1d0c-84a7-42...
5,68,2739609,Colchicum multiflorum,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,d4ecc8ee-655d-42fd-b58d-76c5eae79000
6,54,2739737,Colchicum cupanii,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/2c2873a8-9146-4f...
7,52,2739938,Colchicum neapolitanum,Colchique De Naples,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,f4c0f37a-db7f-43b3-9507-d1d362306108
8,9,2739671,Colchicum montanum,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,q-10447947175
9,1,2739652,Colchicum corsicum,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,45288cbd-b9c3-4d2d-a292-939d1a128fb8


### Afficher les sous clades du clade choisi

In [ ]:
# Liste des clades INPN= ['all','regne', 'classe', 'ordre', 'famille', 'genre','nomScientifique','nomVernaculaire']
# Liste des clades GBIF= ['kingdom', 'class', 'order', 'family', 'genus','species']
clade = 'order'
taxon='Asterales'

explorer_clade(df_biodiv,clade,taxon,cle_ID)

In [ ]:
##A REFAIRE#### Afficher la carte avec les mailles

afficher_carte_maille(carte_maille)

In [ ]:
##A REFAIRE#### Filtrer une liste de mailles à étudier
liste_codes=['10kmL93E089N623']
col_valeur='nombreObs_norm_par_maille_et_regne'

df_filt=df_inpn[df_inpn[cle_geo].isin(liste_codes)]
df_maille=df_filt.groupby(cle_ID)[col_valeur].sum()

df_dico=generer_dictionnaire_taxonomie(df_filt,cle_ID)
df_maille=pd.merge(df_maille,df_dico,on=cle_ID)

afficher_dataframe(df_maille,[col_valeur,cle_ID,'nomVernaculaire','nomScientifique','famille','ordre','classe','regne','especeProtegee'],col_sort=col_valeur).head(10)

In [ ]:
##A REFAIRE#### Rechercher les espèces protégées présentes dans une zone
col_valeur='nombreObs_norm_par_espece'
liste_codes=liste_geo_PNR_Vosges

chercher_especes_protegees(df_inpn,liste_codes,cle_geo='codeMaille10Km',cle_ID='cdRef',col_valeur=col_valeur)

afficher_dataframe(grouped_local_especes_protegee,[col_valeur,cle_ID,'nomVernaculaire','nomScientifique','famille','ordre','classe','regne','especeProtegee']).head(10)

## AFFICHAGE DES DONNEES SOUS FORME DE CARTE 

### Préparation

In [16]:
# Configuration de la carte par défaut
fond_de_carte="Sud"
def charger_couches_SIG_defaut(fig,ax):    
    fig, ax = ajouter_couche_SIG(fig, ax, departement_gpd,
                             facecolor='none',alpha=1,
                             edgecolor="grey",linewidth=1,linestyle='--',
                             with_label=False,col_label='code',label_color="white",fontsize=8
                            )
    
    """ Liste de couches SIG disponibles :
    PNR_gpd_filtered
    border_local_geo
    departement_gpd
    bioregion_gpd
    """
    return fig, ax
    
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap',zoom_manuel=8)
fig, ax = charger_couches_SIG_defaut(fig, ax)
plt.show()

In [ ]:
# Définition des paramètres de la carte
center_x = -3.5E5    # Coordonnée X du centre de la carte
center_y = 6.13E6    # Coordonnée Y du centre de la carte
height = (6.3e6 - center_y) * 2  # Hauteur de la carte
size_x = 15          # Taille de la figure en X
size_y = 10         # Taille de la figure en Y
zoom_size = 8        # Niveau de zoom

# Configuration de la carte avec OpenStreetMap
fig, ax = configurer_carte('OpenStreetMap', center_x, center_y, height, zoom=zoom_size, fig_size=(size_x, size_y))

# Ajout de la couche des départements avec des bordures blanches en pointillés
fig, ax = ajouter_couche_SIG(fig, ax, Zone_gpd_filtered, linewidth=2, edgecolor='black', linestyle='--')

# Affichage de la carte
plt.show()


In [ ]:
# Ajouter une entrée au dictionnaire
nouvelle_carte = "Bretagne"
paramètres = {
    "center_x": center_x,
    "center_y": center_y,
    "height": height,
    "size_x": size_x,
    "size_y": size_y,
    "zoom": 8
}

# Ajouter la nouvelle entrée
dictionnaire_cartes[nouvelle_carte] = paramètres

# Sauvegarder le dictionnaire mis à jour
with open(chemin_dico_cartes, "w", encoding="utf-8") as fichier:
    json.dump(dictionnaire_cartes, fichier, indent=4)

print(f"L'entrée '{nouvelle_carte}' a été ajoutée et enregistrée.")

In [ ]:
(center_x, center_y, height, size_x, size_y, zoom) = (
    dictionnaire_cartes["Afrique de l'Est"].values()
)
print(center_x, center_y, height, size_x, size_y)

In [23]:
# Afficher la liste des instances
instances = list(dictionnaire_cartes.keys())

print("Instances disponibles dans dictionnaire_cartes :")
for instance in instances:
    print("-", instance)

Instances disponibles dans dictionnaire_cartes :
- France
- Madagascar
- PN Cévennes
- Ballons des Vosges
- Allemagne
- Amérique Sud
- Brazil
- Maghreb
- Méditerranée
- Philippines
- Afrique de l'Est
- Chine
- Iran
- Japon+Corée
- Espagne-France-Italie
- Costa Rica
- Monde
- Sri Lanka
- Amérique Centrale
- Asie du Sud-Est
- Asie du Sud+Sud-Est
- Italie
- Espagne
- Europe
- Test
- Sud
- Sud_zoom9
- 88
- Sicily
- Bretagne


### Affichage de la carte pour un taxon

In [24]:
#Afficher la carte de répartition d'un taxon

col_valeur = 'nombreObs' #'nombreObs_norm_par_maille_et_kingdom' ,'nombreObs_unique' , 'nombreObs'
colormap = 'viridis'
#fond_de_carte="Espagne-France-Italie"
log_values = False
SIG_defaut=True
save_carte=False

# Sélection du taxon et du clade
clade = 'species'  # Peut être : nomScientifique, nomVernaculaire, regne, classe, ordre, famille, genre
#taxon = 'Ononis mitissima'
taxon = 'Colchicum filifolium'

seuil=10

# Définition du titre de la figure
titre = f'Carte de répartition de {taxon} dans {zone_name_short}'
#titre = 'Carte de répartition de' #Pour entrée manuelle

# Filtrage des données
df_filt = df_biodiv[df_biodiv[clade] == taxon]

if seuil is not None:
    df_filt = df_filt[df_filt[col_valeur] >= seuil]

# Affichage du fond de carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap',zoom_manuel=8)

# Ajout des données
fig, ax = ajouter_couche_continue(
    fig, ax, df_filt, carte_maille, col_valeur, cle_geo,
    quantile_inf=0.00, quantile_sup=1,
    borne_min=None, borne_max=None,
    cmap_choice=colormap, val_alpha=0.8,
    log_values=log_values
)

if SIG_defaut:
    fig, ax = charger_couches_SIG_defaut(fig, ax)
    
# Ajout de couches SIG supplémentaires
"""
fig, ax = ajouter_couche_SIG(fig, ax, departement_gpd,
                             facecolor='none',alpha=1,
                             edgecolor="grey",linewidth=1,linestyle='--',
                             with_label=True,col_label='code',label_color="white",fontsize=8
                            )
"""

# Ajout du titre et de la légende
ax.set_title(titre, fontsize=16)
fig.text(0.45, 0.15, f'var : {col_valeur}', ha='center', va='center', fontsize=10)

# Affichage de la carte
plt.show()

if save_carte:
    fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI

## INDICE DE BIODIVERSITE et ENDEMISME  

In [29]:
# Configuration des paramètres
col_valeur = 'nombreObs'
colormap = 'plasma'
fond_carte = 'GeoportailSatellite'
n_filt=10
# Préparation des données
df_filt = df_biodiv.copy()  # Filtrer les valeurs NaN sur cle_geo
df_filt = filtrer_top(df_filt, col_valeur, n_filt, cle_ID)  # Filtrer les 10 valeurs dont col > n

# Calcul des indices de biodiversité et d'endémisme
df_indice = calculer_indices(df_filt, col_valeur, cle_geo, cle_ID)

nombre d'espèces retenues dans le df :31266 (55%)


In [38]:
# Paramètrage de la carte à afficher

indice_choisi = "indice_d_endemisme"

fond_de_carte="Ballons des Vosges"
log_values = False
SIG_defaut=True
save=True

"""
Liste des indices : 
nombre_especes
nombre_observations
indice_de_Shannon
indice_de_Simpson
indice_d_endemisme
"""

# Définition du titre de la figure
titre = f"{indice_choisi}_{zone_name_short}"
#titre = 'Carte de ' #Pour entrée manuelle

# Affichage du fond de carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='GeoportailSatellite',zoom_manuel=10)

# Ajout des couches
fig, ax = ajouter_couche_continue(
    fig, ax, df_indice, carte_maille, indice_choisi, cle_geo,
    quantile_inf=0.05, quantile_sup=0.99, borne_min=None, borne_max=None,
    cmap_choice='plasma', val_alpha=0.5, missing_color='black', 
    colorbar_choice=True, log_values=log_values
)

if SIG_defaut:
    fig, ax = charger_couches_SIG_defaut(fig, ax)

# Ajout du titre
ax.set_title(titre, fontsize=16)
fig.text(0.45, 0.15, f'var : {col_valeur}, n_filt : {n_filt}', ha='center', va='center', fontsize=10)
# Affichage de la carte
plt.show()

if save:
    fig.savefig(save_path+'/'+titre+'_'+str(n_filt)+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI


## DETERMINATION DE BIOREGIONS - ANALYSE PAR CLUSTERS GEOGRAPHIQUES

### Préparation

In [45]:
# Analyse en composantes principales

# Paramètres 
col_valeur = 'nombreObs_norm_par_maille_et_kingdom'
predic=True
n_filt=10
log_values = False

#Paramètres de la PCA
variance_threshold=0.99
n_components=None
max_components=1000
df_filt=[]
# Filtrage des données
if predic:
    statut_predic = "avec"
    col_valeur=col_valeur+"_predit"
    df_filt=df_complet_predit.copy()
    
else:
    statut_predic = "sans"
    df_filt=df_biodiv.copy()
    df_filt = filtrer_top(df_filt, 'nombreObs', n_filt, cle_ID)

filtre=None
if filtre is not None:
    df_filt = filtrer_categorie(df_filt, filtre,filtres)
    
print(f"Nombre d'espèces prises en compte : {len(df_filt['speciesKey'].unique())}")

if log_values:
    col_valeur=col_valeur+"_log"

# Réduction de dimension avec PCA
df_pca = analyser_composantes_principales(
    df_filt, cle_geo, col_valeur, cle_ID,
    variance_threshold=variance_threshold, n_components=n_components, max_components=max_components
)

Nombre d'espèces prises en compte : 904
Nombre de composantes nécessaires pour atteindre 99.0% de variance expliquée: 16
Variance expliquée par les dix premières composante: [59.7 17.3 11.4  3.3  2.3  1.4  1.2  0.8  0.4  0.3]
Variance cumulée totale : 99.0 %


In [30]:
# faire varier le nombre de cluster pour en déterminer le nombre optimal
determiner_k(df_pca,n_init=5,max_cluster=20) 

In [46]:
# Clustering 
# Paramètres du clustering
k = 8  # Nombre de clusters
n_init = 30# Nombre de répétitions pour choisir la meilleure solution
methode_clustering = 'kmeans'  # Méthode : 'kmeans' ou 'ward'
display=False

#Paramétrage du critère de contiguité spatiale
λ = 0 # Pénalité spatiale 
methode_contiguite = "euclidean"# Métrique : 'neighbor' ou 'squared euclidean'
if λ == 0:
    methode_contiguite = "None"
parameter = 50  # Paramètre pour la méthode 'neighbor'
n_components = df_pca.shape[1]

# Déterminer les clusters
if λ == 0:
    df_cluster = former_cluster_biogeo(df_pca,cle_geo, method=methode_clustering, k_cluster=k, n_init=n_init,display=display)
elif λ > 0:
    df_cluster = former_cluster_biogeo_avec_critere_spatial(
        df_pca, carte_maille, cle_geo, n_components, k, methode_contiguite, λ, n_init, parameter
    )
else:
    print('Erreur : Lambda doit être >= 0')


### Affichage de la carte des biorégions

In [47]:
# Configuration de la carte
col_valeur_cluster = 'Cluster'
colormap = 'Accent'  # Autres options : 'Spectral_r', 'plasma', 'Paired', 'Set1_r', 'Set2_r', 'Set3', 'tab10', 'Accent' (8)
#fond_de_carte="Italie"
SIG_defaut = True
save_carte = True

# Définition du titre de la figure
countries_name = "_".join(country.replace(" ", "_") for country in countries)

if filtre is not None:
    titre = f"Carte des biorégions de {zone_name_short} - {filtre} {statut_predic} prédiction"
else: titre = f"Carte des biorégions de {zone_name_short} - {statut_predic} prédiction"

# Affichage du fond de carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap') #GeoportailSatellite OpenStreetMap

# Ajout des données
fig, ax = ajouter_couche_discrete(
    fig, ax, df_cluster, carte_maille, col_valeur=col_valeur_cluster, cle_geo=cle_geo,
    cmap_choice=colormap, val_alpha=0.8,
    missing_color=None, legend_choice=True, loc_legend=3, pos_legend=None
)

if SIG_defaut:
    fig, ax = charger_couches_SIG_defaut(fig, ax)

else :
    fig, ax = ajouter_couche_SIG(fig, ax, bioregion_gpd,
                                 facecolor='none',alpha=1,
                                 edgecolor="white",linewidth=1,linestyle='--',
                                 with_label=False,col_label='code',label_color="white",fontsize=8
                                )


# Ajout du titre et de la légende
ax.set_title(titre, fontsize=18)
fig.text(0.5, 0.09, f'variable : {col_valeur}, {cle_geo}, n_pca : {n_components}, méthode de clustering {methode_clustering},k : {k}, λ: {λ}, n_init : {n_init}, méthode de contiguité : {methode_contiguite}',
         ha='center', va='center', fontsize=10)

# Sauvegarde et affichage de la carte
if save_carte:
    # Génération du nom de fichier avec tous les paramètres
    nom_fichier = f"Carte_{zone_name_short}"
    if filtre is not None:
        nom_fichier += f"_{filtre}"
    nom_fichier += f"_{statut_predic}"
    if log_values:
        nom_fichier += "_log"
    nom_fichier += f"_pca{n_components}_predic_k{k}_λ{λ}_{methode_clustering}_{methode_contiguite}.png"
    nom_fichier = nom_fichier.replace(" ", "_")
    fig.savefig(save_path + '/' + nom_fichier + '.png', dpi=300, bbox_inches='tight')

plt.show()

### Cartes des PCA

In [ ]:
# Configuration de la carte des PCA

max_pca=min(5,df_pca.shape[1])
colormap = 'plasma'  # Autres options : 'Spectral_r', 'plasma', 'Paired', 'Set1_r', 'Set2_r', 'Set3', 'tab10', 'Accent' (8)
fond_de_carte="Bretagne"
SIG_defaut = True
save_carte = True

# Définition du titre de la figure
countries_name = "_".join(country.replace(" ", "_") for country in countries)

for i in range(1,max_pca+1):
    col_valeur_PCA = 'PC'+str(i)
    if filtre is not None:
        titre = f"Carte de la PCA {i} de {zone_name_short} - {filtre} {statut_predic} prédiction"
    else: titre = f"Carte de la PCA {i} de {zone_name_short} - {statut_predic} prédiction"
    
    # Affichage du fond de carte
    fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='GeoportailSatellite',zoom_manuel=6) #GeoportailSatellite OpenStreetMap
    
    # Ajout des données
    fig, ax = ajouter_couche_continue(
        fig, ax, df_pca, carte_maille, col_valeur_PCA, cle_geo,
        quantile_inf=0, quantile_sup=1,
        borne_min=None, borne_max=None,
        cmap_choice=colormap, val_alpha=0.8,
        log_values=False
    )
    
    if SIG_defaut:
        fig, ax = charger_couches_SIG_defaut(fig, ax)
    
    
    fig, ax = ajouter_couche_SIG(fig, ax, border_local_geo,
        facecolor='none',alpha=1,
        edgecolor="white",linewidth=1,linestyle='--',
        with_label=False,col_label='code',label_color="white",fontsize=8
        )
    
    
    # Ajout du titre et de la légende
    ax.set_title(titre, fontsize=18)
    fig.text(0.5, 0.09, f'variable : {col_valeur}, n_filt : {n_filt} {cle_geo}, pca_{i}',
             ha='center', va='center', fontsize=10)

        # Sauvegarde et affichage de la carte
    if save_carte:
        # Génération du nom de fichier avec tous les paramètres
        nom_fichier = f"Carte_{zone_name_short}"
        if filtre is not None:
            nom_fichier += f"_{filtre}"
        nom_fichier += f"_{statut_predic}"
        if log_values:
            nom_fichier += "_log"
        nom_fichier += f"_pca{i}_n_filt{n_filt}.png"
        nom_fichier = nom_fichier.replace(" ", "_")
        fig.savefig(save_path + '/' + nom_fichier + '.png', dpi=300, bbox_inches='tight')
    
        plt.show()

In [ ]:
# Pour inverser des composantes :
df_pca['PC2']=-df_pca['PC2']

### Analyse par cluster

In [ ]:
#Statistiques sur les clusters 

# Fusion des données avec les clusters
df_cluster_reset = df_cluster.reset_index()

#df_filt_analyse=df_biodiv.copy() #Si l'on veut reset les filtres
df_filt_analyse=df_filt.copy()

df_merged = pd.merge(df_filt_analyse, df_cluster_reset[[cle_geo, 'Cluster']], how='left')

# Agrégation des données par maille et cluster
df_grouped = (
    df_merged
    .groupby([cle_geo, 'Cluster'], as_index=False)
    .agg({'nombreObs': 'sum', 'nombreObs_unique': 'sum'})
)

# Calcul des statistiques par cluster
cluster_means = df_grouped.groupby('Cluster')[['nombreObs', 'nombreObs_unique']].mean()
cluster_median = df_grouped.groupby('Cluster')[['nombreObs', 'nombreObs_unique']].median()
cluster_quantile1 = df_grouped.groupby('Cluster')[['nombreObs', 'nombreObs_unique']].quantile(0.1)
cluster_quantile9 = df_grouped.groupby('Cluster')[['nombreObs', 'nombreObs_unique']].quantile(0.9)

# Création du tableau de synthèse en concaténant les résultats
stats_summary = pd.concat([cluster_means, cluster_median, cluster_quantile1, cluster_quantile9], axis=1)

# Renommage des colonnes
stats_summary.columns = [
    "Moyenne_nombreObs",
    "Moyenne_nEspece",
    "Médiane_nombreObs",
    "Médiane_nEspece",
    "1er_decile_nombreObs",
    "1er_decile_nEspece",
    "9e_decile_nombreObs",
    "9e_decile_nEspece"
]

# Conversion en entier pour un affichage propre
stats_summary = stats_summary.astype(int)

# Affichage du tableau de synthèse
print("\n", stats_summary)


In [ ]:
# Définition des paramètres
col_valeur = 'nombreObs_norm_par_espece'
seuil_filtrage = 1000  # Seuil pour filtrer les espèces surreprésentées
cluster_choice = 2  # Cluster à afficher

# Filtrage des données
df_filt_analyse = filtrer_top(df_filt_analyse, 'nombreObs', seuil_filtrage, cle_ID)

# Étude de la composition des clusters
cluster_composition = etudier_composition_cluster(df_filt_analyse, df_cluster, dico_taxo, col_valeur, cle_ID, cle_geo)

# Affichage des 50 premières espèces du cluster choisi
df_cluster_choice = cluster_composition[cluster_composition['Cluster'] == cluster_choice]
colonnes_affichage = ['Cluster', col_valeur] + liste_col_taxo

afficher_dataframe(df_cluster_choice, colonnes_affichage, col_valeur).head(10)
afficher_dataframe(df_cluster_choice, ['species','vernacularName_fr','nombreObs_norm_par_espece'], col_valeur).head(10)

## ANALYSE PAR CLUSTERS D'ESPECES

### Préparation

In [ ]:
# ---------------------------------------------------------------
# 1. Filtrage des espèces pour réduire le temps de calcul
# ---------------------------------------------------------------
n_filt = 200
df_global = filtrer_top(df_biodiv, 'nombreObs_norm_par_maille_et_kingdom', n_filt, cle_ID)
colonnes_obs = [col for col in df_global.columns if 'nombreObs' in col]
col_val = 'nombreObs_norm_par_maille_et_kingdom'
methode_corr = 'pearson'  # pearson, kendall
import_mat_corr = True
calculate_mat_corr = False
save_mat_corr = False

path_data_mat_corr = os.path.join(data_path, "mat_corr")
filename_mat_corr=f'mat_corr_{zone_name_short}_{"".join(regnes_a_garder)}_{n_filt}_{methode_corr}_{cle_geo}.csv'

if import_mat_corr:
    # Chargement des données
    mat_corr = pd.read_csv(os.path.join(path_data_mat_corr, filename_mat_corr), dtype=str)
    mat_corr = mat_corr.apply(pd.to_numeric, errors='coerce')
    
elif calculate_mat_corr:
    # Jalon 1 : début du processus
    start_time = time.time()
    mat_corr = calculer_matrice_correlation(df_global, col_val, methode_corr, cle_geo, cle_ID)
    mat_corr = mat_corr.astype(float)
    end_time = time.time()
    print(f"Temps écoulé: {end_time - start_time:.2f} secondes")

# ---------------------------------------------------------------
# 2. Calcul de la matrice de corrélation et génération du dendrogramme
# ---------------------------------------------------------------
if save_mat_corr:
    # Sauvegarde de la matrice de corrélation
    mat_corr_tosave = pd.DataFrame(mat_corr)
    mat_corr_tosave.to_csv(os.path.join(path_data_mat_corr, filename_mat_corr), index=False)


In [ ]:
# ---------------------------------------------------------------
# 3.Génération du dendrogramme
# ---------------------------------------------------------------
methode_dendogram='ward' #  ward, complete
Z = generer_dendogram(mat_corr, methode='ward', display=0)  

In [ ]:
# ---------------------------------------------------------------
# 4. Formation des clusters en fonction du niveau choisi
# ---------------------------------------------------------------
col_val = 'nombreObs_norm_par_maille_et_kingdom'
lvl = 8  # Niveau de découpage du dendrogramme
criterion = 'distance'

df_especes_cluster = former_cluster_espece(df_global, Z, col_valeur=col_val, level=lvl, crit=criterion, cle_ID=cle_ID, cle_geo=cle_geo)
df_global_cluster = pd.merge(df_global, df_especes_cluster[[cle_ID, 'Cluster_corr']], on=cle_ID)
df_global_cluster['Cluster_corr'] = pd.to_numeric(df_global_cluster['Cluster_corr'], errors='coerce').astype('Int64')

print(f'Nombre de clusters formés : {len(df_especes_cluster["Cluster_corr"].unique())}')


### Recherche d'une espèce

In [ ]:
# ---------------------------------------------------------------
# 5. Recherche et affichage des espèces du cluster d'une espèce donnée
# ---------------------------------------------------------------
espece = 'Epuraea variegata'
save_fig = False
cle_sujet = df_global[df_global['species'] == espece][cle_ID].unique()[0]

# Recherche du cluster contenant l'espèce
num_cluster = chercher_numcluster_espece(df_especes_cluster, cle_ID, cle_sujet)
df_global_cluster['Cluster_corr'] = pd.to_numeric(df_global_cluster['Cluster_corr'], errors='coerce').astype('Int64')

# Liste des espèces du cluster sélectionné, triée selon col_valeur
liste_especes_cluster_choisi = lister_especes_dans_cluster(df_global_cluster, num_cluster,colonnes_obs,col_valeur=col_val, cle_ID=cle_ID)

print(f'Le cluster regroupe {len(liste_especes_cluster_choisi)} espèces')

# Affichage du cluster
afficher_dataframe(liste_especes_cluster_choisi, [col_val] + liste_col_taxo, col_sort=col_val, n_rows=10)


In [ ]:
# ---------------------------------------------------------------
# 6. Affichage de l'aire de répartition du cluster sélectionné
# ---------------------------------------------------------------
colormap = 'viridis'
fond_de_carte = "France"
titre = f'{col_val} - Cluster numéro {num_cluster}'

df_filt = df_global_cluster[df_global_cluster['Cluster_corr'] == num_cluster]

quantile_inf = 0.0
df_filt = df_filt[df_filt[col_val] > df_filt[col_val].quantile(quantile_inf, interpolation='linear')]

# Configuration de la carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap')

# Ajout des couches cartographiques
fig, ax = ajouter_couche_continue(fig, ax, df_filt, carte_maille, col_val, cle_geo,
                                  quantile_inf=0.0, quantile_sup=0.99,
                                  cmap_choice=colormap, val_alpha=0.75, colorbar_choice=True)

fig, ax = ajouter_couche_SIG(fig, ax, border_local_geo, linewidth=1, edgecolor='grey', linestyle='--')

# Ajout du titre et des annotations
fig.text(0.5, 0.09, f'col_corr : {col_val}', ha='center', va='center', fontsize=10)
ax.set_title(titre, fontsize=16)

if save_fig:
    # Sauvegarde de la figure
    nom_fichier = f"Carte_{zone_name_short}_clusternum{num_cluster}.png"
    nom_fichier = nom_fichier.replace(" ", "_")
    fig.savefig(f'{save_path}/{nom_fichier}.png', dpi=300, bbox_inches='tight')

plt.show()


### Etude d'une sous-zone

In [ ]:
# ---------------------------------------------------------------
# 7. Sélection des codes de maille pour un site donné
# ---------------------------------------------------------------
parc_gpd = PN_et_PNR_gpd
nom_parc = 'Ballons des Vosges'
cle_nom_site = 'NOM_SITE'
methode_nom = 'contains' # contains exact
taux_min=0.6 # Taux de la maille qui doit être compris dans la zone : 1= 100% de la maille dans la zone

liste_codes = lister_mailles_dans_site(carte_maille, parc_gpd, nom_parc, taux_min=taux_min, cle_geo=cle_geo,
                                       cle_nom_site=cle_nom_site, methode=methode_nom)
df_global_raw=df_global.copy()
print(f"Nombre de mailles dans la zone choisie : {len(liste_codes)}")

In [ ]:
# ---------------------------------------------------------------
# 8. Filtrer les mailles présentes dans liste_codes et grouper par cluster
# ---------------------------------------------------------------
col_choice = 'nombreObs_norm_par_espece' 
crit='sum'
nmin=200 #Si on veut filtrer les espèces les + présentes
crit_norm=True


df_global=filtrer_top(df_global_raw,var='nombreObs',nmin=nmin,cle_ID=cle_ID)
df_global_cluster = pd.merge(df_global, df_especes_cluster[[cle_ID, 'Cluster_corr']], on=cle_ID)
df_global_cluster['Cluster_corr'] = pd.to_numeric(df_global_cluster['Cluster_corr'], errors='coerce').astype('Int64')


df_local = df_global[df_global[cle_geo].isin(liste_codes)]
df_grouper_par_cluster_local = grouper_par_cluster(df_local, df_especes_cluster, colonnes_obs,col_choice, cle_geo=cle_geo, cle_ID=cle_ID,crit=crit,norm=crit_norm)
afficher_dataframe(df_grouper_par_cluster_local, ['Cluster_corr', 'species', 'vernacularName_fr',
                                                  'nombreObs', 'nombreObs_norm_par_espece',
                                                  'nombreObs_norm_par_maille_et_kingdom'],
                   col_sort=col_choice).head(10)


In [ ]:
# ---------------------------------------------------------------
# 9. Afficher les détails du cluster choisi dans la zone 
# ---------------------------------------------------------------

# Paramétrage
rank_cluster = 0
col_abondance = 'nombreObs'  # Pour les abondances brutes
col_critere = 'nombreObs_norm_par_espece'  # Pour le critère de tri

# Récupération du numéro du cluster sélectionné
num_cluster = int(df_grouper_par_cluster_local['Cluster_corr'].iloc[rank_cluster])
print(f'Cluster n° {num_cluster}')

# Préparation du DataFrame local avec l'information de cluster
df_local_cluster = pd.merge(df_local, df_especes_cluster[[cle_ID, 'Cluster_corr']], on=cle_ID)
df_local_cluster['Cluster_corr'] = pd.to_numeric(df_local_cluster['Cluster_corr'], errors='coerce').astype('Int64')

# Étude du cluster local via la fonction
resultat = etudier_un_cluster_local(
    df_global_cluster, 
    df_local_cluster, 
    cle_ID=cle_ID, 
    num_cluster=num_cluster,
    colonnes_obs=colonnes_obs,
    col_choice_1=col_abondance,
    col_choice_2=col_critere
)

# Affichage du résultat (en dehors de la fonction comme tu l'as voulu)
print(f'Le cluster regroupe {len(resultat)} espèces')
afficher_dataframe(resultat, [col_abondance, col_critere] + liste_col_taxo, col_sort=col_critere, n_rows=10)


In [ ]:
# ---------------------------------------------------------------
# 9. Afficher les espèces absentes dans la zone étudiée mais les plus susceptibles d'y être présentes
# ---------------------------------------------------------------
max_cluster=df_local_cluster['Cluster_corr'].nunique()

resultat=rechercher_especes_localement_absentes(df_global_cluster, df_local_cluster,df_grouper_par_cluster_local, cle_ID,colonnes_obs, col_choice='nombreObs',seuil=None,max_cluster=min(max_cluster,50))

afficher_dataframe(resultat, ['num_cluster'] + liste_col_taxo,n_rows=10)

## ANALYSE PAR CORRELATION

### Entre un sujet et un objet

In [47]:
# Définition des paramètres
clade = 'species' 
sujet = 'Colchicum filifolium'
col_corr = 'nombreObs_norm_par_maille_et_kingdom'
methode_choice = 'kendall'  # Options: 'pearson', 'kendall', 'spearman'
col_filt = 'nombreObs_norm_par_maille_et_kingdom'
obs_min=100

# Filtrage et préparation des données
df_global = filtrer_top(df_biodiv, col_filt, obs_min, cle_ID)
df_global = filtrer_categorie(df_global, 'plantes',filtres)

df_sujet = df_biodiv[df_biodiv[clade] == sujet]

# Combinaison et nettoyage des données
df_global = pd.concat([df_sujet, df_global]).drop_duplicates()
df_global_complet = completer_df(df_global, df_global, cle_geo, cle_ID)
df_global_complet = df_global_complet.sort_values(by=cle_ID)

# Calcul des corrélations avec mesure du temps
t_start = time.time()
df_corr = calculer_correlation_sujet(
    df_global_complet, col_corr, clade, sujet, 
    methode=methode_choice, cle_ID=cle_ID, cle_geo=cle_geo
)
t_end = time.time()
print(f"Temps d'exécution : {t_end - t_start:.2f} secondes")

# Affichage des résultats
afficher_dataframe(df_corr, ['Coeff_corr'] + liste_col_taxo, col_sort='Coeff_corr').head(10)

nombre d'espèces retenues dans le df :2367 (6%)
Temps d'exécution : 11.63 secondes


,Coeff_corr,speciesKey,species,vernacularName_fr,vernacularName_en,genus,family,order,class,phylum,kingdom,occurrenceID
0,1.00,2739964,Colchicum filifolium,NaN,NaN,Colchicum,Colchicaceae,Liliales,Liliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/b0be1d0c-84a7-42...
1,0.14,2792833,Ophrys bertolonii,NaN,Bertoloni'S Bee Orchid,Ophrys,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,7de5e723-af56-441b-8764-d8131f988edc
2,0.13,7331330,Helianthemum marifolium,NaN,NaN,Helianthemum,Cistaceae,Malvales,Magnoliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/a8f05cff-cf92-40...
3,0.13,2792725,Ophrys flavicans,NaN,NaN,Ophrys,Orchidaceae,Asparagales,Liliopsida,Tracheophyta,Plantae,https://www.inaturalist.org/observations/16093...
4,0.12,3069761,Euphorbia paralias,Euphorbe Maritime,Sea Spurge,Euphorbia,Euphorbiaceae,Malpighiales,Magnoliopsida,Tracheophyta,Plantae,https://www.inaturalist.org/observations/72681622
5,0.11,2965379,Medicago arborea,Luzerne Arborescente,Tree Medick,Medicago,Fabaceae,Fabales,Magnoliopsida,Tracheophyta,Plantae,d8dfe5ce-d413-4eab-a0a4-944cea8bd4e4
6,0.11,3925918,Anemone hortensis,NaN,NaN,Anemone,Ranunculaceae,Ranunculales,Magnoliopsida,Tracheophyta,Plantae,5b8d83e5-690c-4fa6-9279-87e5d7058987
7,0.11,5348659,Coronilla juncea,NaN,NaN,Coronilla,Fabaceae,Fabales,Magnoliopsida,Tracheophyta,Plantae,https://www.inaturalist.org/observations/24717...
8,0.10,3632234,Crithmum maritimum,Fenouil Marin,Rock Samphire,Crithmum,Apiaceae,Apiales,Magnoliopsida,Tracheophyta,Plantae,https://www.inaturalist.org/observations/10601...
9,0.10,3894066,Teucrium pseudochamaepitys,NaN,NaN,Teucrium,Lamiaceae,Lamiales,Magnoliopsida,Tracheophyta,Plantae,http://flore.silene.eu/occtax/9c9a0198-cb60-4b...


In [ ]:
# Filtrer les résultats
df_filt = filtrer_categorie(df_corr, 'plantes',filtres)
afficher_dataframe(df_filt,['Coeff_corr']+liste_col_taxo,col_sort='Coeff_corr').head(10)

In [ ]:
df_filt[df_filt['species']=='Lotus corniculatus']

In [ ]:
# Paramètres de l'affichage
colormap = 'plasma'
SIG_defaut = True
save_carte=False
log_values=False
affichage_zone_recensee=False

with_specie = 1  # 1 : prend en compte l'espèce, 0 : ne la prend pas en compte
n_sigma = 0  # 0 : seuil = 0
seuil_quantile = 0.0
seuil_observation = 1

# Identification de l'espèce cible
cle_sujet = df_biodiv[df_biodiv['species'] == sujet][cle_ID].unique()[0]

# Calcul de la prédiction
col_recalcul = 'nombreObs_norm_par_maille_et_kingdom'
colonne_resultat = f"{col_recalcul}_predit"
df_sujet_predit = recalculer_nombreObs_par_correlation(
    df_global, df_corr, col_recalcul, cle_sujet, with_specie,
    cle_ID=cle_ID, cle_geo=cle_geo
)

# Définition du titre
titre = f"Aire de répartition potentielle de {sujet}"

# Détermination du seuil de prédiction
df_global_avec_presence = df_global[
    (df_global['species'] == sujet) & (df_global['nombreObs'] >= seuil_observation)
]
liste_mailles_avec_sujet = df_global_avec_presence[cle_geo].unique()
df_predit_avec_presence = df_sujet_predit[df_sujet_predit[cle_geo].isin(liste_mailles_avec_sujet)]
seuil_prediction = df_predit_avec_presence[colonne_resultat].quantile(seuil_quantile)
df_filt = df_sujet_predit[df_sujet_predit[colonne_resultat] >= seuil_prediction]

# Affichage du fond de carte
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap', zoom_manuel=6)

# Ajout des données prédites
fig, ax = ajouter_couche_continue(
    fig, ax, df_filt, carte_maille, colonne_resultat, cle_geo,
    quantile_inf=0.01, quantile_sup=0.99,
    borne_min=None, borne_max=None,
    cmap_choice=colormap, val_alpha=0.8,
    log_values=log_values
)

# Ajout des couches SIG par défaut si activé
if SIG_defaut:
    fig, ax = charger_couches_SIG_defaut(fig, ax)

# Ajout des points de présence réelle
if affichage_zone_recensee:
    df_global_avec_presence = df_global[(df_global['species'] == sujet) & (df_global['nombreObs'] >= seuil_observation)]
    fig, ax = ajouter_couche_point(
        fig, ax, df_global_avec_presence, carte_maille,
        col_valeur='nombreObs', cle_geo=cle_geo,
        color_dot='black', size_dot=2, legend_choice=True
    )

# Ajout du titre et informations complémentaires
ax.set_title(titre, fontsize=16)
fig.text(0.5, 0.09, 
         f'Méthode de corr : {methode_choice}, col_corr : {col_valeur}, col_recalcul : {col_recalcul}, with_specie : {with_specie}, sigma={n_sigma}',
         ha='center', va='center', fontsize=10)

# Sauvegarde de la carte
if save_carte:
    fig.savefig(f"{save_path}/{titre}.png", dpi=300, bbox_inches='tight')

# Affichage de la carte
plt.show()

In [ ]:
# Affichage et regression des données prédites en fonction des données d'observation
col_values_corr='nombreObs_norm_par_maille_et_kingdom'
colonne_predit = f"{col_values_corr}_predit"
x, y = prepare_data(df_global, df_sujet_predit, cle_geo, sujet, col_values_corr)
coefficients = fit_and_plot(x, y, col_values_corr, sujet, loi='all')
plot_residuals(x, y, coefficients['linear'][0], coefficients['linear'][1])


### Extrapolation pour toutes les espèces

In [29]:
# ---------------------------------------------------------------
# 1. Filtrage des espèces pour réduire le temps de calcul
# ---------------------------------------------------------------
n_filt = 1000
df_global = filtrer_top(df_biodiv, 'nombreObs', n_filt, cle_ID)
zone_name_short="Sud de la France"
col_val = 'nombreObs_norm_par_maille_et_kingdom'
methode_corr = 'kendall'  # pearson, kendall
import_mat_corr = True
calculate_mat_corr = False
save_mat_corr = False

path_data_mat_corr = os.path.join(data_path, "mat_corr")
filename_mat_corr=f'mat_corr_{zone_name_short}_{"".join(regnes_a_garder)}_{n_filt}_{col_val}_{methode_corr}_{cle_geo}.csv'

# ---------------------------------------------------------------
# 2. Calcul ou importation de la matrice de corrélation
# ---------------------------------------------------------------
if import_mat_corr:
    # Chargement des données
    mat_corr = pd.read_csv(os.path.join(path_data_mat_corr, filename_mat_corr), dtype=str)
    print("✅ Importation terminée")
    
elif calculate_mat_corr:
    # Jalon 1 : début du processus
    start_time = time.time()
    mat_corr = calculer_matrice_correlation(df_global, col_val, methode_corr, cle_geo, cle_ID)
    mat_corr = mat_corr.astype(float)
    end_time = time.time()
    print(f"Temps écoulé: {end_time - start_time:.2f} secondes")
    print("✅ Calcul terminé")

if save_mat_corr:
    # Sauvegarde de la matrice de corrélation
    mat_corr_tosave = pd.DataFrame(mat_corr)
    mat_corr_tosave.to_csv(os.path.join(path_data_mat_corr, filename_mat_corr), index=False)
    print("✅ Sauvegarde terminée")


nombre d'espèces retenues dans le df :3763 (9%)


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\MANTIS\\Data\\mat_corr\\mat_corr_Sud de la France_AnimaliaPlantae_1000_nombreObs_norm_par_maille_et_kingdom_kendall_codeMaille5Km.csv'

In [31]:
# ---------------------------------------------------------------
# 3. Complétion du DataFrame et calcul de la prédiction
# ---------------------------------------------------------------
import_df_predit = True
by_chunk=True

calculate_df_predit=False
save_df_predit = False
col_val_predit=f"{col_val}_predit"

path_data_df_predit = os.path.join(data_path, "df_predit")
filename_df_predit=f'df_predit_{zone_name_short}_{"".join(regnes_a_garder)}_{n_filt}_{col_val}_{methode_corr}_{cle_geo}.csv'

df_complet_predit=[]
if import_df_predit:
    if by_chunk:
        chunk_size = 100000  # Nombre de lignes par chunk
        chunks = pd.read_csv(os.path.join(path_data_df_predit, filename_df_predit),chunksize=chunk_size,low_memory=False)
        df_complet_predit = pd.concat(chunks, ignore_index=True)
    else:
        df_complet_predit = pd.read_csv(os.path.join(path_data_df_predit, filename_df_predit),low_memory=False)
    print("✅ Importation terminée")

elif calculate_df_predit:
    # Complétion du DataFrame
    liste_codes_complet = df_global[cle_geo].unique()
    df_global_complet = completer_df(df_global, df_global, cle_geo, cle_ID)
    df_global_complet = df_global_complet.sort_values(by=cle_ID, ascending=True)

    # Jalon 2 : début du processus de prédiction
    start_time = time.time()
    df_complet_predit = calculer_prediction(df_global_complet, mat_corr, col_val, cle_geo, cle_ID)

    df_biodiv_complet = df_biodiv.drop(columns=[col_val_predit], errors='ignore')
    df_biodiv_complet = pd.merge(df_biodiv_complet, df_complet_predit[[cle_geo, cle_ID, col_val_predit]], on=[cle_geo, cle_ID])

    # Calculer et afficher le temps écoulé
    end_time = time.time()
    print(f"Temps écoulé: {end_time - start_time:.2f} secondes")

    # Normalisation par espèce
    df_complet_predit = normaliser_par_espece(df_complet_predit, cle_ID, col_val_predit)
    df_complet_predit = normaliser_log(df_complet_predit, col_val_predit)

    print("✅ Calcul et normalisations terminés")
    
if save_df_predit:
    # Sauvegarde des prédictions
    path_data_predit = os.path.join(data_path, "df_predit")
    df_tosave = pd.DataFrame(df_complet_predit)
    df_tosave.to_csv(os.path.join(path_data_df_predit, filename_df_predit), index=False)
    print("✅ Sauvegarde terminée")


✅ Importation terminée


In [40]:
# ---------------------------------------------------------------
# 4. Configuration de la carte
# ---------------------------------------------------------------

#fond_de_carte = "Italie"
colormap = 'plasma'
n_sigma = 2  # 1 pour animal, 2 pour plantes
seuil_observation = 10
zoom_size = 8
save = True

# Sélection de l'espèce
espece = 'Colchicum filifolium'
cle_sujet = df_biodiv[df_biodiv['species'] == espece][cle_ID].unique()[0]
col_values_corr = 'nombreObs_norm_par_maille_et_kingdom'
colonne_predit = f"{col_values_corr}_predit"

titre = f'Aire de répartition potentielle de {espece}'

# Filtrage des données
df_complet_predit_espece = df_complet_predit[df_complet_predit['species'] == espece]
df_global_avec_presence = df_global[(df_global['species'] == espece) & (df_global['nombreObs'] >= seuil_observation)]
liste_mailles_avec_espece = df_global_avec_presence[cle_geo].unique()
df_complet_predit_avec_presence = df_complet_predit_espece[df_complet_predit_espece[cle_geo].isin(liste_mailles_avec_espece)]

# Calcul du seuil de prédiction
seuil_prediction = calculer_seuil(
    df_global, df_complet_predit_espece, cle_sujet, col_values_corr, seuil_observation,
    n_sigma, cle_ID=cle_ID, cle_geo=cle_geo
)

df_filt = df_complet_predit_espece[df_complet_predit_espece[colonne_predit] >= seuil_prediction]

# ---------------------------------------------------------------
# 5. Affichage de la carte
# ---------------------------------------------------------------
fig, ax = afficher_fond_carte(fond_de_carte, dictionnaire_cartes, source_fond='OpenStreetMap')

# Ajout des données
fig, ax = ajouter_couche_continue(
    fig, ax, df_filt, carte_maille, colonne_predit, cle_geo,
    quantile_inf=0, quantile_sup=1,
    cmap_choice=colormap, val_alpha=0.75, colorbar_choice=True
)

# Ajout des points de présence
fig, ax = ajouter_couche_point(
    fig, ax, df_global_avec_presence, carte_maille, col_valeur='nombreObs',
    cle_geo=cle_geo, color_dot='black', size_dot=1, legend_choice=True
)

# Ajout de couches SIG
#fig, ax = ajouter_couche_SIG(fig, ax, departement_gpd, linewidth=1, edgecolor='grey', linestyle='--')

# Ajout du titre et des annotations
ax.set_title(titre, fontsize=16)
fig.text(
    0.5, 0.09, 
    f'Méthode de corr : {methode_corr}, col_corr : {col_values_corr}, col_recalcul : {col_values_corr}, sigma={n_sigma}',
    ha='center', va='center', fontsize=10
)

# Sauvegarde et affichage
if save:
    fig.savefig(f"{save_path}/{titre}.png", dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
# ---------------------------------------------------------------
# 6. Regressions des données prédites en fonction des données d'observation
# ---------------------------------------------------------------
col_values_corr='nombreObs_norm_par_maille_et_kingdom'
colonne_predit = f"{col_values_corr}_predit"
x, y = prepare_data(df_global, df_complet_predit_espece, cle_geo, espece, col_values_corr)
coefficients = fit_and_plot(x, y, col_values_corr, espece, loi='all')
plot_residuals(x, y, coefficients['linear'][0], coefficients['linear'][1])


In [ ]:
# ---------------------------------------------------------------
# 7. Conversion en données quantitatives et affichage de la carte pour le taxon choisi
# ---------------------------------------------------------------

df_filt=df_complet_predit_espece.copy()
methode='linear'

colonne_predit_modelisation='nombreObs_predit_glm_'+methode
df_filt_trans = appliquer_transformation(df_filt, colonne_predit, methode, coefficients)
df_filt_trans = df_filt_trans.dropna(subset=[colonne_predit_modelisation])  # Remove NaNs in the column
df_filt_trans=df_filt_trans[df_filt_trans[colonne_predit_modelisation]>=0]
#fig, ax=configurer_carte('OpenStreetMap',center_x,center_y,height,zoom=zoom_size,fig_size=(size_x, size_y))
fig, ax = afficher_fond_carte(zone_name_short, dictionnaire_cartes, source_fond='OpenStreetMap')
fig, ax=ajouter_couche_continue(fig, ax, df_filt_trans,carte_maille,colonne_predit_modelisation,cle_geo,quantile_inf=0.0,quantile_sup=1,
                                cmap_choice=colormap,val_alpha=0.75,colorbar_choice=True)
#fig, ax=ajouter_couche_point(fig, ax,df_global_avec_presence,carte_maille,col_valeur='nombreObs',cle_geo=cle_geo,color_dot='black',size_dot=2,legend_choice=True)
fig, ax=ajouter_couche_SIG(fig, ax,border_local_geo,linewidth=1,edgecolor='grey',linestyle='--')
fig.text(0.5, 0.09, f'Méthode de corr : {methode}, col_corr : {col_values_corr}, col_recalcul : {col_values_corr}, sigma={n_sigma}',
         ha='center', va='center', fontsize=10)
#fig, ax=ajouter_couche_SIG(fig, ax,departement_gpd)
titre = f'Aire de répartition prédite de {espece} - modèle {methode}'
ax.set_title(titre, fontsize=16)  # Taille de la police définie à 16
fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI

plt.show()

### Recherche des espèces potentiellement présentes dans une sous-zone

In [ ]:
# ---------------------------------------------------------------
# 1. Sélection des codes de maille pour un site donné
# ---------------------------------------------------------------
mode='mailles' #zone ou mailles

if mode == 'zone':
    parc_gpd = PN_et_PNR_gpd #geodataframe contenant la geometry de la zone à étudier : PN_et_PNR_gpd departement_gpd
    cle_nom_site = 'NOM_SITE' #nom de la colonne contenant le nom/clé de la zone à étudier : NOM_SITE code
    nom_parc = 'Ballons des Vosges' #nom/clé de la zone à étudier :  Ballons des Vosges 88 
    methode_nom = 'contains' # contains ou exact
    taux_min=1# Taux de la maille qui doit être compris dans la zone : 1= 100% de la maille dans la zone
    liste_codes = lister_mailles_dans_site(carte_maille, parc_gpd, nom_parc, taux_min=taux_min, cle_geo=cle_geo,
                                           cle_nom_site=cle_nom_site, methode=methode_nom)

elif mode=='mailles':
    liste_codes=['10kmE01393N03795ITA']

print(f"Nombre de mailles dans la zone choisie : {len(liste_codes)}")

In [ ]:
# ---------------------------------------------------------------
# 2. Prédiction au niveau local
# ---------------------------------------------------------------
col_values_prediction='nombreObs_norm_par_maille_et_kingdom' # 'nombreObs_norm_par_maille_et_kingdom' 'nombreObs' 'nombreObs_norm_par_espece'

# Complétion du DataFrame
df_local=df_global[(df_global[cle_geo].isin(liste_codes))]
df_local_complet=completer_df(df_local,df_global,cle_geo,cle_ID)
df_local_complet = df_local_complet.sort_values(by=cle_ID,ascending=True)

# Jalon 2 : début du processus de prédiction
df_local_complet_predit=calculer_prediction(df_local_complet, mat_corr, col_values_prediction,cle_geo,cle_ID)
colonne_predit = f"{col_values_corr}_predit"

# Normalisation par espèce
df_local_complet_predit = normaliser_par_espece(df_local_complet_predit, cle_ID, colonne_predit)
df_local_complet_predit = normaliser_log(df_local_complet_predit, colonne_predit)

In [ ]:
# ---------------------------------------------------------------
# 3. Recherche des espèces suceptibles d'être présentes au niveau local
# ---------------------------------------------------------------
filtre='arthropodes'
df_filt = filtrer_categorie(df_local_complet_predit, filtre,filtres)
df_local_especes_absentes=recherche_espece_absente(df_filt,colonne_predit,cle_geo,cle_ID)

print(f"Nombre d'espèces : {df_local_especes_absentes.shape[0]}")

afficher_dataframe(df_local_especes_absentes,[colonne_predit]+liste_col_taxo,col_sort=colonne_predit).head(50)

In [ ]:
nom_espece = "Pamphagus marmoratus"

# Trouver les indices où l'espèce est absente
rang = df_local_especes_absentes.index[df_local_especes_absentes['species'] == nom_espece].tolist()

# Nombre total d'espèces absentes
total = df_local_especes_absentes.shape[0]

if rang:
    print(f"L'espèce '{nom_espece}' est absente à la position {rang[0]} sur {total} espèces.")
else:
    print(f"L'espèce '{nom_espece}' n'est pas présente dans la liste des espèces absentes.")


## SUIVI TEMPOREL

In [ ]:
# Suivi des espèces (ou autres clade) disparues entre période 1 et 2
df_filt=df_biodiv_periode
df_suivi_mailles = suivre_disparition_geo(df_filt,cle_ID,cle_geo)

col_valeur='taux_apparue'
titre=col_valeur
colormap='YlOrRd_r'
fond_carte='GeoportailSatellite'
fig, ax = afficher_fond_carte(zone_name, dictionnaire_cartes, source_fond='OpenStreetMap')
fig, ax=ajouter_couche_continue(fig, ax, df_suivi_mailles,carte_maille,col_valeur,cle_geo,
                                quantile_inf=0.0,quantile_sup=0.99,
                                cmap_choice='viridis',val_alpha=0.7,
                               log_values=False)
fig, ax=ajouter_couche_SIG(fig, ax,border_local_geo,linewidth=1,edgecolor='grey',linestyle='--')
ax.set_title(titre, fontsize=18)  # Taille de la police définie à 16
# Ajoute une ligne de texte en dessous de la figure
fig.text(0.45, 0.15, f'var : {col_valeur}', ha='center', va='center', fontsize=10)
ax.set_title(titre, fontsize=16)  # Taille de la police définie à 16
#fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI
plt.show()

In [ ]:
col_valeur='taux_disparue'
titre=col_valeur
colormap='YlOrRd'
fond_carte='GeoportailSatellite'
fig, ax=configurer_carte('OpenStreetMap',center_x,center_y,height,zoom=zoom_size,fig_size=(size_x, size_y))
fig, ax=ajouter_couche_continue(fig, ax, df_suivi_mailles,carte_maille,col_valeur,cle_geo,
                                quantile_inf=0.0,quantile_sup=0.99,
                                cmap_choice='viridis',val_alpha=0.7,
                               log_values=False)
fig, ax=ajouter_couche_SIG(fig, ax,border_local_geo,linewidth=1,edgecolor='grey',linestyle='--')
ax.set_title(titre, fontsize=18)  # Taille de la police définie à 16
# Ajoute une ligne de texte en dessous de la figure
fig.text(0.45, 0.15, f'var : {col_valeur}', ha='center', va='center', fontsize=10)
ax.set_title(titre, fontsize=16)  # Taille de la police définie à 16
#fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI

plt.show()

In [ ]:
# Afficher la carte de l'évolution de l'aire de répartition d'un taxon
taxon='Arenaria provincialis'
clade='species'
df_filt=df_biodiv_periode[(df_biodiv_periode[clade]==taxon)]
df_filt=df_filt[[cle_geo, clade,'periode']].drop_duplicates().reset_index(drop=True)
grouped = df_filt.groupby([cle_geo, clade])['periode'].apply(list).reset_index()

# Appliquer la fonction pour créer la colonne 'statut'
grouped['statut'] = grouped['periode'].apply(determiner_statut)
# Garder seulement les colonnes nécessaires
final_df = grouped[[cle_geo, clade, 'statut']]

titre=f"Statut de {taxon} en Europe du Sud-Ouest"
            
statut_colors = {
    "Colonisation 1991-2010": "green",
    "Colonisation 2011-2024": "lightgreen",
    "Disparition 1801-1990": "red",
    "Disparition 1991-2011": "orange",
    "Présence 1991-2010 mais absent aujourd'hui": "yellow",
    "Présence continue jusqu'à aujourd'hui": "blue"
}

fig, ax = afficher_fond_carte(zone_name, dictionnaire_cartes, source_fond='OpenStreetMap')
fig, ax=ajouter_couche_statut(fig, ax, final_df, carte_maille, statut_colors,col_valeur='statut',cle_geo=cle_geo, val_alpha=0.75, legend_choice=True,loc_legend="upper right")
fig, ax=ajouter_couche_SIG(fig, ax,border_local_geo,linewidth=1,edgecolor='black',linestyle='-')
#fig, ax=ajouter_couche_SIG(fig, ax,departement_gpd)
ax.set_title(titre, fontsize=18)  # Taille de la police définie à 16

fig.savefig(save_path+'/'+titre+'.png', dpi=300, bbox_inches='tight')  # Enregistre au format PNG avec une résolution de 300 DPI

plt.show()